# 01 — Kalshi Public API: Exotics Metadata Exploration

**Goal:** Understand how Kalshi *itself* defines and structures exotic / combo / multivariate (MVE) markets, *before* we audit any derived label on Dune.

**Why this comes first:** Our Dune dashboard uses `kalshi.market_report.category = 'Exotics'`. That `category` field is a derived label — assigned upstream by someone (Kalshi? a Dune curator?). If we don't know what it derives *from*, we can't tell whether the dashboard is right or wrong. Step one is always: **anchor the definition at the source**.

**Methodology pattern used throughout this notebook:**
1. **Question** — what do we want to know?
2. **Code** — minimal call to find out
3. **Observation** — what the response actually told us (good or bad)
4. **Implication** — what this means for the dashboard methodology

We do NOT draw conclusions in code comments. Conclusions live in markdown so they survive re-runs.


## 0. Setup

In [1]:
import requests
import pandas as pd
import json
from pprint import pprint

# Two candidate hosts for Kalshi's public API. We test both.
HOSTS = [
    "https://api.elections.kalshi.com/trade-api/v2",
    "https://trading-api.kalshi.com/trade-api/v2",
]

def get(host, path, **params):
    """Tiny wrapper: returns (status_code, json_or_text)."""
    r = requests.get(f"{host}{path}", params=params, timeout=15)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, r.text[:500]


## 1. Q: Is the API reachable, and on which host?

**Observation (filled in after run):**
- `api.elections.kalshi.com/trade-api/v2` → HTTP **200** ✅
- `trading-api.kalshi.com/trade-api/v2` → HTTP **401** (auth required)

**Implication:** The elections host is our public, unauthenticated source of truth for market metadata. Use it for every call below.


In [2]:
for host in HOSTS:
    code_, body = get(host, "/markets", limit=1)
    print(f"{host}  →  HTTP {code_}")
    if code_ == 200 and isinstance(body, dict):
        print("   keys:", list(body.keys()))


https://api.elections.kalshi.com/trade-api/v2  →  HTTP 200
   keys: ['cursor', 'markets']
https://trading-api.kalshi.com/trade-api/v2  →  HTTP 401


**Observation:** [filled in after run]
**Implication:** the host that returns 200 is our base for everything below.


## 2. Q: What does a *generic* (non-exotic) market look like?

**Observation (filled in after run):** Markets have **44 fields**. The endpoint returned an MVE as the first market (not a true baseline — Kalshi's market table is dominated by MVE rows), but the response shape is clear. Key universal-looking fields: `ticker`, `event_ticker`, `market_type`, `notional_value_dollars`, `volume_fp`, `liquidity_dollars`, status, bid/ask, etc.

**Caveat:** we did NOT get a true non-MVE baseline here. To get one, filter to a non-MVE series (e.g. a single sports market). Adding to the open-questions list.

**Implication:** standard market fields exist on every market. The interesting ones for our methodology are the **MVE-specific fields** seen below.


In [3]:
BASE = "https://api.elections.kalshi.com/trade-api/v2"  # set after Section 1

status, body = get(BASE, "/markets", limit=1, status="open")
print("HTTP:", status)
if isinstance(body, dict) and body.get("markets"):
    m = body["markets"][0]
    print("Number of fields on a market:", len(m))
    print()
    pprint(m)


HTTP: 200
Number of fields on a market: 44

{'can_close_early': True,
 'close_time': '2026-05-17T17:10:00Z',
 'created_time': '2026-05-14T12:20:29.594485Z',
 'custom_strike': {'Associated Events': 'KXMLBGAME-26MAY141310DETNYM,KXMLBSPREAD-26MAY141235COLPIT,KXMLBTOTAL-26MAY141340MIAMIN',
                   'Associated Market Sides': 'yes,yes,no',
                   'Associated Markets': 'KXMLBGAME-26MAY141310DETNYM-NYM,KXMLBSPREAD-26MAY141235COLPIT-PIT2,KXMLBTOTAL-26MAY141340MIAMIN-9',
                   'Multivariate Event Ticker': 'KXMVESPORTSMULTIGAMEEXTENDED-S202633BA5111CA6'},
 'event_ticker': 'KXMVESPORTSMULTIGAMEEXTENDED-S202633BA5111CA6',
 'expected_expiration_time': '2026-05-14T20:40:00Z',
 'expiration_time': '2026-05-17T17:10:00Z',
 'expiration_value': '',
 'fractional_trading_enabled': True,
 'is_provisional': True,
 'last_price_dollars': '0.0000',
 'latest_expiration_time': '2026-05-17T17:10:00Z',
 'liquidity_dollars': '0.0000',
 'market_type': 'binary',
 'mve_collection_tick

**Observation:** [filled in after run]
**Implication:** these fields are our baseline schema. Anything an exotic has *beyond* this is structurally meaningful.


## 3. Q: What does an *exotic* market look like (KXMVE prefix)?

**Observation (filled in after run):** Direct API filter `?series_ticker=KXMVECROSSCATEGORY` works → returned 20 markets in one call. Two MVE-specific fields jump out (not present on standard markets):

- **`mve_collection_ticker`** — e.g. `KXMVECROSSCATEGORY-R` — the parlay *template*
- **`mve_selected_legs`** — an **explicit array** of the legs in this parlay, each entry: `{ event_ticker, market_ticker, side }`

Example: one cross-category parlay had 5 legs spanning 3 MLB games and 2 NHL games, all on the `yes` side.

**Implication: THIS IS THE GROUND-TRUTH LEG METADATA WE NEEDED.** We can compute the leg count of every parlay as `len(mve_selected_legs)`. This is the dimension-table field that lets us answer Evan Semet's 1-vs-N question against any Dune dataset.


In [4]:
# Try direct series filter
status, body = get(BASE, "/markets", limit=20, series_ticker="KXMVECROSSCATEGORY")
print("Direct series_ticker filter — HTTP:", status)
if isinstance(body, dict):
    mkts = body.get("markets", [])
    print(f"Returned {len(mkts)} markets")
    if mkts:
        print()
        pprint(mkts[0])


Direct series_ticker filter — HTTP: 200
Returned 20 markets

{'can_close_early': True,
 'close_time': '2026-05-17T16:35:00Z',
 'created_time': '2026-05-14T12:20:55.238838Z',
 'custom_strike': {'Associated Events': 'KXMLBGAME-26MAY141235COLPIT,KXMLBGAME-26MAY141240WSHCIN,KXMLBGAME-26MAY141310DETNYM,KXNHLGAME-26MAY14MTLBUF,KXNHLGAME-26MAY14VGKANA',
                   'Associated Market Sides': 'yes,yes,yes,yes,yes',
                   'Associated Markets': 'KXMLBGAME-26MAY141235COLPIT-COL,KXMLBGAME-26MAY141240WSHCIN-WSH,KXMLBGAME-26MAY141310DETNYM-NYM,KXNHLGAME-26MAY14MTLBUF-MTL,KXNHLGAME-26MAY14VGKANA-ANA',
                   'Multivariate Event Ticker': 'KXMVECROSSCATEGORY-S2026A2B81042B86'},
 'event_ticker': 'KXMVECROSSCATEGORY-S2026A2B81042B86',
 'expected_expiration_time': '2026-05-15T04:30:00Z',
 'expiration_time': '2026-05-17T16:35:00Z',
 'expiration_value': '',
 'fractional_trading_enabled': True,
 'is_provisional': True,
 'last_price_dollars': '0.0000',
 'latest_expiration_time'

## 4. Q: Does Kalshi expose a `series` resource — and does it carry the exotic flag?

**Observation (filled in after run):** Yes. `GET /series/KXMVECROSSCATEGORY` returns:

```json
{
  "ticker": "KXMVECROSSCATEGORY",
  "title": "MVE Cross Category",
  "category": "Exotics",      ← ground-truth label
  "frequency": "custom",
  "fee_type": "quadratic",
  "fee_multiplier": 1,
  ...
}
```

**Implication:** The `category = 'Exotics'` label on Dune's `kalshi.market_report` is **sourced from Kalshi's own series.category field**. The dashboard label is well-founded *in principle*. What we still need to verify:
1. Does *every* `KXMVE`-prefixed series have `category = 'Exotics'`?
2. Are there series with `category = 'Exotics'` that do NOT start with `KXMVE`?
3. Does Dune's rollup pick all of them up?


In [5]:
status, body = get(BASE, "/series/KXMVECROSSCATEGORY")
print("HTTP:", status)
if isinstance(body, dict):
    pprint(body)


HTTP: 200
{'series': {'additional_prohibitions': ['Persons who are employed by any of '
                                        'the Source Agencies are not permitted '
                                        'to trade on the Contract.',
                                        'Persons who hold any material, '
                                        'non-public information on the '
                                        'Underlying are not permitted to trade '
                                        'on the Contract.'],
            'category': 'Exotics',
            'contract_terms_url': 'https://kalshi-public-docs.s3.amazonaws.com/contract_terms/FOOTBALLSTATS.pdf',
            'contract_url': 'https://kalshi-public-docs.s3.us-east-1.amazonaws.com/regulatory/product-certifications/FOOTBALLSTATS.pdf',
            'fee_multiplier': 1,
            'fee_type': 'quadratic',
            'frequency': 'custom',
            'last_updated_ts': '2026-03-09T14:05:04.379097Z',
            'product

**Observation:** [filled in after run]
**Implication:**
- If `body.series.category == "Exotics"` → category is a typed field, dashboard label is well-founded
- If category is missing or generic → the dashboard label was assigned by something downstream (Dune curator), and we need to find out by what rule


## 5. Q: For a combo/parlay, are the legs exposed explicitly?

**Observation (filled in after run):** YES — fully exposed via `mve_selected_legs`. The full field list on an MVE market includes:

```
mve_collection_ticker      → KXMVECROSSCATEGORY-R   (the template / family)
mve_selected_legs          → [{ event_ticker, market_ticker, side }, ...]
event_ticker               → KXMVECROSSCATEGORY-S2026A2B81042B86 (this parlay's event)
ticker                     → KXMVECROSSCATEGORY-S2026A2B81042B86-782608235F5 (this specific market)
custom_strike              → dict mirroring the legs as comma-separated strings
```

**Crucial structural insight:** Each parlay configuration is **its own market with its own ticker**. When a user trades a 5-leg parlay, they trade ONE contract on ONE ticker (`KXMVECROSSCATEGORY-S...-...`). The legs are *metadata describing what the parlay refers to*, NOT separate tradable markets that get hit individually.

**Implication for the 1-vs-N question:** Structurally, `kalshi.trade_report` filtered to `report_ticker LIKE 'KXMVE%'` should show **one trade per parlay-bundle transaction**, not N trades. The leg count expands the *meaning* of the trade, not its *row count*. This means:
- If Dune's row count = 1 per trade → methodology is "1-contract counting", and the dashboard's $3.47B is the legitimate parlay-bundle figure
- If Sam McQuillan's $8.5B over 5 months is correct, **he is almost certainly counting legs** (multiplying volume by `len(mve_selected_legs)`)

We still need to confirm this empirically against Dune's `kalshi.trade_report` schema (next notebook).


In [6]:
# Pull one specific exotic market and inspect ALL fields for leg-ish structure
status, body = get(BASE, "/markets", limit=1, series_ticker="KXMVECROSSCATEGORY")
if isinstance(body, dict) and body.get("markets"):
    m = body["markets"][0]
    print("All field names on this exotic market:")
    for k in sorted(m.keys()):
        v = m[k]
        v_preview = json.dumps(v)[:80] if not isinstance(v, str) else v[:80]
        print(f"  {k:30s}  →  {v_preview}")


All field names on this exotic market:
  can_close_early                 →  true
  close_time                      →  2026-05-17T16:35:00Z
  created_time                    →  2026-05-14T12:20:55.238838Z
  custom_strike                   →  {"Associated Events": "KXMLBGAME-26MAY141235COLPIT,KXMLBGAME-26MAY141240WSHCIN,K
  event_ticker                    →  KXMVECROSSCATEGORY-S2026A2B81042B86
  expected_expiration_time        →  2026-05-15T04:30:00Z
  expiration_time                 →  2026-05-17T16:35:00Z
  expiration_value                →  
  fractional_trading_enabled      →  true
  is_provisional                  →  true
  last_price_dollars              →  0.0000
  latest_expiration_time          →  2026-05-17T16:35:00Z
  liquidity_dollars               →  0.0000
  market_type                     →  binary
  mve_collection_ticker           →  KXMVECROSSCATEGORY-R
  mve_selected_legs               →  [{"event_ticker": "KXMLBGAME-26MAY141235COLPIT", "market_ticker": "KXMLBGAME-26M
 

## 6. Q: Build a metadata DataFrame of all KXMVE-prefixed markets

**Observation (filled in after run):** Pagination hit the safety cap at page 50 with cumulative **~50k KXMVE markets** still open. Most pages were ~100% KXMVE-prefixed, meaning Kalshi's open-market table is *dominated* by auto-generated parlay configurations. Each potential 3/4/5-leg combination becomes its own market.

This explains why the dashboard's exotic counts can look enormous in raw market-count terms even if traded volume is modest.

**Implication:** The dimension table we want has ~50k+ rows. We should:
1. Persist this snapshot as a CSV (joinable to Dune trade data)
2. Compute `leg_count = len(mve_selected_legs)` and tag the subtype prefix on each row
3. Keep an eye on `is_provisional` — many of these are draft / never-traded markets, which inflates the universe count


In [7]:
# Page through all open markets, collect KXMVE-prefixed ones
all_kxmve = []
cursor = None
page = 0
while True:
    params = dict(limit=1000, status="open")
    if cursor:
        params["cursor"] = cursor
    status, body = get(BASE, "/markets", **params)
    if status != 200 or not isinstance(body, dict):
        print("STOPPED at page", page, "status", status)
        break
    mkts = body.get("markets", [])
    kxmve_in_page = [m for m in mkts if m.get("ticker","").startswith("KXMVE")]
    all_kxmve.extend(kxmve_in_page)
    page += 1
    cursor = body.get("cursor")
    print(f"page {page}: total markets {len(mkts)}, KXMVE here {len(kxmve_in_page)}, cumulative KXMVE {len(all_kxmve)}, cursor={'yes' if cursor else 'no'}")
    if not cursor or page > 50:  # safety cap
        break

df = pd.DataFrame(all_kxmve)
print()
print("Final KXMVE count:", len(df))
if len(df):
    print("Columns:", list(df.columns))
    print("Unique series tickers:", df['event_ticker'].nunique() if 'event_ticker' in df else 'n/a')
df.head()


page 1: total markets 1000, KXMVE here 1000, cumulative KXMVE 1000, cursor=yes
page 2: total markets 1000, KXMVE here 1000, cumulative KXMVE 2000, cursor=yes
page 3: total markets 1000, KXMVE here 994, cumulative KXMVE 2994, cursor=yes
page 4: total markets 1000, KXMVE here 1000, cumulative KXMVE 3994, cursor=yes
page 5: total markets 1000, KXMVE here 883, cumulative KXMVE 4877, cursor=yes
page 6: total markets 1000, KXMVE here 1000, cumulative KXMVE 5877, cursor=yes
page 7: total markets 1000, KXMVE here 1000, cumulative KXMVE 6877, cursor=yes
page 8: total markets 1000, KXMVE here 1000, cumulative KXMVE 7877, cursor=yes
page 9: total markets 1000, KXMVE here 992, cumulative KXMVE 8869, cursor=yes
page 10: total markets 1000, KXMVE here 1000, cumulative KXMVE 9869, cursor=yes
page 11: total markets 1000, KXMVE here 994, cumulative KXMVE 10863, cursor=yes
page 12: total markets 1000, KXMVE here 1000, cumulative KXMVE 11863, cursor=yes
page 13: total markets 1000, KXMVE here 1000, cumul

,can_close_early,close_time,created_time,custom_strike,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,is_provisional,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2026-05-17T16:35:00Z,2026-05-14T12:20:55.238838Z,{'Associated Events': 'KXMLBGAME-26MAY141235CO...,KXMVECROSSCATEGORY-S2026A2B81042B86,2026-05-15T04:30:00Z,2026-05-17T16:35:00Z,,True,True,...,KXMVECROSSCATEGORY-S2026A2B81042B86-782608235F5,"yes Colorado,yes Washington,yes New York M,yes...",2026-05-14T12:20:55.834396Z,0.00,0.00,0.0000,0.00,0.0000,0.00,"yes Colorado,yes Washington,yes New York M,yes..."
1,True,2026-05-17T16:35:00Z,2026-05-14T12:20:55.023012Z,{'Associated Events': 'KXMLBGAME-26MAY141235CO...,KXMVESPORTSMULTIGAMEEXTENDED-S2026DB289D0064E,2026-05-15T05:10:00Z,2026-05-17T16:35:00Z,,True,True,...,KXMVESPORTSMULTIGAMEEXTENDED-S2026DB289D0064E-...,"yes Pittsburgh,yes Cincinnati,yes New York M,y...",2026-05-14T12:20:55.834396Z,0.00,0.00,0.0000,0.00,0.0000,0.00,"yes Pittsburgh,yes Cincinnati,yes New York M,y..."
2,True,2026-05-29T23:00:00Z,2026-05-14T12:20:54.658141Z,"{'Associated Events': 'KXNBAAST-26MAY15DETCLE,...",KXMVECROSSCATEGORY-S202686E5099790A,2026-05-16T02:00:00Z,2026-05-29T23:00:00Z,,True,True,...,KXMVECROSSCATEGORY-S202686E5099790A-2547326F851,"yes James Harden: 6+,yes Donovan Mitchell: 20+...",2026-05-14T12:20:54.83463Z,0.00,0.00,0.0000,0.00,0.0000,0.00,"yes James Harden: 6+,yes Donovan Mitchell: 20+..."
3,True,2026-05-31T04:00:00Z,2026-05-14T12:20:54.560164Z,"{'Associated Events': 'KXPGAMAKECUT-PGC26,KXPG...",KXMVESPORTSMULTIGAMEEXTENDED-S20262D6D9B1A610,2026-05-16T02:00:00Z,2026-05-31T04:00:00Z,,True,True,...,KXMVESPORTSMULTIGAMEEXTENDED-S20262D6D9B1A610-...,"yes Billy Horschel,yes Cameron Young,yes Justi...",2026-05-14T12:20:54.83463Z,0.00,0.00,0.0000,0.00,0.0000,0.00,"yes Billy Horschel,yes Cameron Young,yes Justi..."
4,True,2026-05-17T16:35:00Z,2026-05-14T12:20:54.472112Z,{'Associated Events': 'KXMLBTOTAL-26MAY141235C...,KXMVESPORTSMULTIGAMEEXTENDED-S2026F8E37F5F33B,2026-05-15T02:15:00Z,2026-05-17T16:35:00Z,,True,True,...,KXMVESPORTSMULTIGAMEEXTENDED-S2026F8E37F5F33B-...,"yes Over 7.5 runs scored,yes Over 5.5 runs sco...",2026-05-14T12:20:54.83463Z,0.00,0.00,0.0000,0.00,0.0000,0.00,"yes Over 7.5 runs scored,yes Over 5.5 runs sco..."


## 7. What we learned — open questions parking lot

**Definitively answered:**
- [x] **Q1 (reachability)** — `api.elections.kalshi.com/trade-api/v2` is the public host
- [x] **Q3 (exotic shape)** — exotic markets carry `mve_collection_ticker` and `mve_selected_legs`
- [x] **Q4 (series-level category)** — Kalshi's own `series.category = 'Exotics'` is the source of the label
- [x] **Q5 (legs exposed?)** — YES, via `mve_selected_legs`. Each entry has `event_ticker`, `market_ticker`, `side`
- [x] **Q6 (universe size)** — ~50,000+ KXMVE markets currently *open*; dominated by auto-generated parlay configurations

**Structural conclusions worth highlighting:**
1. Each parlay configuration is its **own tradable market** with its own ticker. Legs are metadata, not separately-traded entities. → strongly suggests Dune trade rows are 1-per-bundle, not N-per-bundle.
2. The dashboard's `category='Exotics'` label is **sourced from Kalshi**, not invented downstream. Good provenance.
3. The all-time dashboard figure ($3.47B) being smaller than Sam's 5-month claim ($8.5B) now has a probable explanation: **Sam is multiplying by `len(mve_selected_legs)`** (leg-counting). To be confirmed.

**Still open — feed into notebook 02:**
- [ ] Pull a true non-MVE baseline market for clean comparison
- [ ] Confirm every `KXMVE*` series has `category='Exotics'` (and vice versa)
- [ ] Compute `leg_count` distribution across the 50k KXMVE markets (mean, median, max)
- [ ] Tag each market with its subtype family (`KXMVECROSSCATEGORY`, `KXMVESPORTSMULTIGAMEEXTENDED`, `KXMVEOSCARS`, `KXMVECBCHAMPIONSHIP`, ...)
- [ ] Persist the metadata DataFrame to CSV for joining to Dune
- [ ] **Confirm against Dune `kalshi.trade_report` schema** that one trade = one row (not N rows per leg). This is the empirical confirmation of the structural conclusion.
- [ ] Reconcile the dashboard's $3.47B all-time exotic volume against Sam McQuillan's monthly figures, broken down by leg count
